# World Model FineTuning Trainer

> Finetune the world model on a combination of real + corrupted messsages and csi.

In [ ]:
#| default_exp trainers.wm_tune

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
import torch
import os
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import wandb
import hydra
from pathlib import Path
from fastcore.utils import patch
from loguru import logger
from omegaconf import DictConfig
from einops import rearrange
import torch.nn.functional as F
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer
from c3jepa_wm.utils import channel, channel_optimal, compute_power_schedule, apply_channel


In [ ]:
#| export
class TrainerScheduler:
    def __init__(self, wm_optimizer):
        self.wm_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            wm_optimizer, mode='min', factor=0.5, patience=5
        )

    def step(self, val_loss):
        self.wm_scheduler.step(val_loss['val_jepa_loss'])
        
    

In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 wm_lr=1e-4, 
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.wm_lr = wm_lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

    def init_optimizer(self, model, lr, weight_decay=0.01):
        return torch.optim.AdamW(
                list(model.parameters()),
                lr=lr,
                weight_decay=weight_decay
            )
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

In [ ]:
#| export
class WMFinetuner(BaseTrainer):
    def __init__(self, data_module, model, device, slurm_jobid, wm_lr,
                 history_size, num_preds, lambda_sigreg, early_stop_patience=15, **kwargs):
        super().__init__(
            data_module=data_module, device=device, wm_lr=wm_lr,
            slurm_jobid=slurm_jobid, **kwargs
        )
        self.history_size = history_size
        self.num_preds = num_preds
        self.lambda_sigreg = lambda_sigreg
        self.early_stop_patience = early_stop_patience
        self.best_val_loss = float("inf")
        self.epochs_since_improvement = 0

        self.model = model["jepa"].to(device)
        self.vqvae = model["vqvae"].to(device)
        self.wm_optimizer = self.init_optimizer(self.model, lr=self.wm_lr, weight_decay=1e-3)
        self.scheduler = TrainerScheduler(self.wm_optimizer)

        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(
            project_name=self.project_name, ckp_dir=self.ckp_dir,
            slurm_jobid=self.slurm_jobid, agent_id="WM_trainer", rank=0, n_best=3
        )
        
        self.model.predictor.msg_token_embedding.weight.data[:self.vqvae.vq_layer.K].copy_(
            self.vqvae.vq_layer.embedding.detach()
        )
        


In [ ]:
#| export
@patch
def fit(self: WMFinetuner, cfg: DictConfig):
    for epoch in range(1, cfg.pipeline.max_epochs + 1):
        train_loss = self.train_epoch(epoch)
        metrics = self.evaluate_epoch(epoch)
        self.scheduler.step(metrics)
        self.checkpoint(epoch, metrics)

        val_loss = metrics["val_jepa_loss"]
        if val_loss < self.best_val_loss - 1e-4:   # small min-delta to avoid noise-triggered resets
            self.best_val_loss = val_loss
            self.epochs_since_improvement = 0
        else:
            self.epochs_since_improvement += 1

        if self.epochs_since_improvement >= self.early_stop_patience:
            logger.info(f"Early stopping at epoch {epoch}: no improvement for {self.early_stop_patience} epochs.")
            break
        
        

In [ ]:
#| export
@patch
@torch.no_grad()
def get_msg_indices(self: WMFinetuner, sender_pov_seq):
    """Helper function to get message indices from the pretrained VQ-VAE for a given sender POV sequence.
    Args:
        sender_pov_seq: Tensor of shape (B, T, C, H, W) representing the sender's point-of-view image sequence.
    Returns:
        msg_indices: Tensor of shape (B, T, 49) 
    """
    B, T, C, H, W = sender_pov_seq.shape
    sender_pov_flat = rearrange(sender_pov_seq.to(self.device), "b t c h w -> (b t) c h w")  # (B*T, C, H, W)
    msg_indices_flat = self.vqvae.get_message_indices(sender_pov_flat)  # (B*T, 7, 7)
    msg_indices = rearrange(msg_indices_flat, "(b t) h w -> b t (h w)", b=B, t=T)   # (B, T, 49)
    return msg_indices



In [ ]:
#| export
@patch
def train_epoch(self: WMFinetuner, epoch):
    self.model.train()

    total_loss_jepa = 0.0
    perfect_comm = 1
    for batch_idx, batch in enumerate(self.train_loader):
        perfect_comm = 1 if batch_idx % 2 == 0 else 0

        batch = {k: v.to(self.device) for k, v in batch.items()}   
        jepa_loss = self.train_batch(epoch, batch, perfect_comm=perfect_comm)
        total_loss_jepa += jepa_loss

        logger.info(f"Completed batch {batch_idx+1}/{len(self.train_loader)} for epoch {epoch} with JEPA loss {jepa_loss:.4f}")

    avg_loss_jepa = total_loss_jepa / len(self.train_loader)

    logger.info(f"Epoch [{epoch}/{self.epochs}] - Train JEPA Loss: {avg_loss_jepa:.4f}")

    return avg_loss_jepa


In [ ]:
#| export
@patch
def train_batch(self: WMFinetuner, epoch, batch, perfect_comm):
    B = batch["sender_pov"].shape[0]
    T = self.history_size
    self.wm_optimizer.zero_grad()

    # 1. Get message indices from pretrained VQ-VAE and get csi from the batch
    msg_indices = self.get_msg_indices(
            batch["sender_pov"]
        )  # (B, T, 49)   
    # import ipdb; ipdb.set_trace()
    msg_indices_flat = rearrange(
        msg_indices, "b t d -> (b t) d"
    )
    
    csi = batch["sender_csi"] # (B, T, nighbours = 1, dim= 1)
    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )
    received_msg = channel_optimal(msg_indices_flat, csi_flat, device= self.device, perfect_comm= perfect_comm)
    received_msg = rearrange(received_msg, "(b t) d -> b t d", b=B, t=T)

    csi = csi.squeeze(-2).squeeze(-1)  # (B, T, 1, 1) -> (B, T)

    output = self.model.encode(batch)

    emb = output["emb"]
    tgt_emb = emb[:, self.num_preds:]
    act_emb = output["act_emb"].reshape(B, T, -1)

    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = received_msg[:, :T]          # perfect message
    ctx_csi = csi[:, :T]                  # perfect csi

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg, ctx_csi)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    loss_dict = {k: v for k, v in output.items() if "loss" in k}
    wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

    output['jepa_loss'].backward()
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    self.wm_optimizer.step()
    return output['jepa_loss'].item()


In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_epoch(self: WMFinetuner, epoch):
    self.model.eval()

    all_metrics = {}
    perfect_comm = 1
    for batch_idx, batch in enumerate(self.val_loader):
        batch = {k: v.to(self.device) if torch.is_tensor(v) else v 
                for k, v in batch.items()}
        perfect_comm = 1 if batch_idx % 2 == 0 else 0

        metrics = self.evaluate_batch(batch, perfect_comm=perfect_comm)

        for k, v in metrics.items():
            all_metrics.setdefault(k, []).append(v)
        
    # Average over all batches
    avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
    wandb.log({**avg_metrics, "epoch": epoch})
    return avg_metrics


In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_batch(self: WMFinetuner, batch, perfect_comm):
    B = batch["sender_pov"].shape[0]
    T = self.history_size

    msg_indices = self.get_msg_indices(
            batch["sender_pov"]
        )  # (B, T, 49)   
    msg_indices_flat = rearrange(
        msg_indices, "b t d -> (b t) d"
    )
    
    csi = batch["sender_csi"] # (B, T, nighbours = 1, dim= 1)
    csi_flat = rearrange(
        batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
    )
    received_msg = channel_optimal(msg_indices_flat, csi_flat, device= self.device, perfect_comm= perfect_comm)
    received_msg = rearrange(received_msg, "(b t) d -> b t d", b=B, t=T)

    csi = csi.squeeze(-2).squeeze(-1)  # (B, T, 1, 1) -> (B, T)

    output = self.model.encode(batch)
    
    emb = output["emb"]
    tgt_emb = emb[:, self.num_preds:]
    act_emb = output["act_emb"].reshape(B, T, -1)

    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    ctx_msg = received_msg[:, :T]          # perfect message
    ctx_csi = csi[:, :T]                  # perfect csi

    pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg, ctx_csi)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    return {
        "val_jepa_loss": output['jepa_loss'].item(),
    }


In [ ]:
#| export
@patch
def checkpoint(self: WMFinetuner, epoch, val_loss):

    checkpoint_state = {
        "epoch": epoch,
        "wm_optimizer_state_dict": self.wm_optimizer.state_dict(),
        "wm_model_state_dict": self.model.state_dict(),
        "val_jepa_loss": val_loss['val_jepa_loss']
        }
    
    acc = -val_loss["val_jepa_loss"]
    
    self.ck_pointer.save_checkpoint(state= checkpoint_state, current_acc= acc, step= epoch)
    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()